In [ ]:
import torch
import blackbox_model

import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import configs





import FMfuncs
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt


import PGFM
import PGFM_perturb

In [ ]:
2

In [ ]:
##%%
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

record = np.load('./saved_model/Apr23RLFM_advs207_iter_train_record.npz') 
# RLFM_advs206_woref_iter_train_record
# Apr22RLFM_advs207_wref_iter_train_record
loss_record = record['constraint_loss_record']
in_prob_record = record['adv_record']
rwd_record = record['flow_loss_record']

##%%
plt.plot(loss_record)
plt.grid()
plt.xlabel('x10 iterations')
plt.ylabel('loss')
plt.show()

plt.plot(in_prob_record)
plt.grid()
plt.xlabel('x10 iterations')
plt.ylabel('adv num')
plt.show()

plt.plot(rwd_record)
plt.grid()
plt.xlabel('x10 iterations')
plt.ylabel('rwd')
plt.show()

In [ ]:
bb_model = blackbox_model.black_box_model_class()

In [ ]:
from torchvision.datasets.mnist import MNIST
data_test = MNIST('./data',
                  train=False,
                  download=True,
                  transform=transforms.Compose([
                      transforms.Resize((32, 32)),
                      transforms.ToTensor()]))
data_loader = torch.utils.data.DataLoader(data_test,
                                          batch_size=10000,
                                          shuffle=False)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

correct = 0
total = 0

data_all = None
label_all = None

with torch.no_grad():
    for inputs, labels in data_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        predicted = bb_model.predict(inputs)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        if data_all is None:
            data_all = inputs
            label_all = labels
        else:
            data_all = torch.cat((data_all, inputs), dim=0)
            label_all = torch.cat((label_all, labels), dim=0)

test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")
data_all = data_all.to(configs.device)
label_all = label_all.to(configs.device)

In [ ]:
train_rate = 0.8

adv_data_train = data_all[:int(train_rate * len(data_all))]
adv_data_test = data_all[int(train_rate * len(data_all)):]
adv_label_train = label_all[:int(train_rate * len(label_all))]
adv_label_test = label_all[int(train_rate * len(label_all)):]

In [ ]:

FM_class = FMfuncs.OTFlowMatching()
# PGFM_class = RLFMfuncs.RLFM(bb_model)
PGFM_class = PGFM.PGFM(bb_model)
PGFMp_class = PGFM_perturb.PGFM_perturb(bb_model)

In [ ]:
a = torch.randn(10,2)

In [ ]:
torch.min(a,dim=1)[0]

In [ ]:
# ckpt1 = torch.load('./saved_model/FM_adv_iter_20000.pth', map_location=configs.device)
ckpt2 = torch.load('./saved_model/Apr23RLFM_advs207_iter_train_730000.pth', map_location=configs.device)
# FMworef_100000
# Apr22RLFM_advs207_wref_iter_train_300000: apr23 record, t0=0.7
# Apr23RLFM_advs207_iter_train_500000_record: t0=0.8, s=15
# Apr23RLFM_advs207_iter_train_730000
stage2model = PGFM_class.policy
# stage2model = PGFM_class.get_untrained_model_stage2_var()
stage2model.load_state_dict(ckpt2)
# res = RLFMclass.RLFMsample(stage1model, stage2model, 1000, mode = 'train2')
res = PGFM_class.RLFMsample( stage2model, adv_data_test,
                   default_stage1t = 0.8, default_RLstep_S = 30) # from_scratch train2 configs.default_stage1_t

res = torch.clip(res, 0, 1)
# res = FMfuncs.sampler(stage1model, inp, stoptime=1, default_generation_step = 100)





# res = adv_data_test
l2norm = torch.norm(res - adv_data_test, p=2, dim = (1,2,3))
print('Mean l2norm:', l2norm.mean().item())


with torch.no_grad():
    # for inputs, labels in data_loader:
    inputs, labels = res.to(device), adv_label_test.to(device)
    predicted = bb_model.predict(inputs)
    total = labels.size(0)
    correct = (predicted == labels).sum().item()

test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")

row = 3
num = 3

success_ind = torch.where(predicted!=labels)[0].cpu().numpy()
# success_ind = torch.where((predicted != labels) & (labels == 6))[0].cpu().numpy()
# rand_ind = success_ind[np.random.randint(0,len(success_ind), row*num)]
rand_ind = success_ind[np.random.choice(len(success_ind), size=row*num, replace=False)]
i=0
plt.figure(figsize=(12,6))
for j in rand_ind:
    # j = j.item()
    fig1 = adv_data_test[j,0].cpu().numpy()
    fig2 = res[j,0].cpu().numpy()
    i+=1
    plt.subplot(row,num*2,i)
    plt.title(adv_label_test[j].item())
    plt.imshow(fig1, cmap='gray')

    plt.axis('off')
    i+=1
    plt.subplot(row,num*2,i)
    plt.title(predicted[j].item())
    plt.imshow(fig2, cmap='gray')
    plt.axis('off')
plt.show()